# System Design & Applied ML

Technical reference for video generation ML engineering: backprop from scratch, system design exercises, and self-assessment cards.

5 key areas:
1. **Math fundamentals** — manual forward/backward pass through a transformer block
2. **System design** — video inference serving, data pipelines, re-captioning at scale
3. **Architecture knowledge** — diffusion vs AR, MoE, attention variants
4. **Data pipeline depth** — end-to-end video data, quality filtering, sharding
5. **Industry awareness** — open-source landscape, competitive positioning

In [ ]:
# EXERCISE: Manual Transformer Block Forward Pass
# Implement each step from raw weights — no nn.Module, no autograd.

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

# Transformer block parameters
d_model = 64
n_heads = 4
d_head = d_model // n_heads
d_ff = 256
seq_len = 8
batch_size = 1

# Initialize weights
W_q = torch.randn(d_model, d_model) * 0.02
W_k = torch.randn(d_model, d_model) * 0.02
W_v = torch.randn(d_model, d_model) * 0.02
W_o = torch.randn(d_model, d_model) * 0.02
W_1 = torch.randn(d_model, d_ff) * 0.02
b_1 = torch.zeros(d_ff)
W_2 = torch.randn(d_ff, d_model) * 0.02
b_2 = torch.zeros(d_model)
gamma_1 = torch.ones(d_model)
beta_1 = torch.zeros(d_model)
gamma_2 = torch.ones(d_model)
beta_2 = torch.zeros(d_model)

# Input
x = torch.randn(batch_size, seq_len, d_model)

print("=== Manual Transformer Block Forward Pass ===\n")

# YOUR CODE: implement all 4 steps manually
# Step 1: Layer Norm (mean/var along last dim, apply gamma/beta)
# Step 2: Multi-Head Self-Attention (project Q/K/V, reshape to heads,
#          scaled dot-product, reshape back, output projection, residual)
# Step 3: Layer Norm 2
# Step 4: FFN (linear → GELU → linear, residual)
#
# Print shapes at each step. Final output should be (1, 8, 64).

raise NotImplementedError

In [ ]:
# Now compute gradients manually for the attention portion
print("=== Manual Backward Pass (Attention) ===\n")

# Use autograd-enabled tensors for verification
x_ag = x.clone().requires_grad_(True)
W_q_ag = W_q.clone().requires_grad_(True)
W_k_ag = W_k.clone().requires_grad_(True)
W_v_ag = W_v.clone().requires_grad_(True)

# Forward pass (attention only)
Q_ag = x_ag @ W_q_ag
K_ag = x_ag @ W_k_ag
V_ag = x_ag @ W_v_ag

scores = Q_ag @ K_ag.transpose(-2, -1) / (d_model ** 0.5)
weights = F.softmax(scores, dim=-1)
attn_out = weights @ V_ag

# Simple loss: sum of outputs
loss = attn_out.sum()

# Manual backward through attention:
print("Backward through: loss = sum(softmax(QK^T/√d) @ V)")
print()

# d_loss/d_attn_out = ones (since loss = sum)
d_attn_out = torch.ones_like(attn_out)
print(f"d_loss/d_attn_out: all ones, shape {d_attn_out.shape}")

# d_loss/d_weights = d_attn_out @ V^T
d_weights = d_attn_out @ V_ag.transpose(-2, -1).detach()
print(f"d_loss/d_weights: shape {d_weights.shape}")

# d_loss/d_V = weights^T @ d_attn_out
d_V = weights.detach().transpose(-2, -1) @ d_attn_out
print(f"d_loss/d_V: shape {d_V.shape}")

# Softmax Jacobian: d_softmax/d_scores = diag(s) - s @ s^T
# Applied: d_scores = weights * (d_weights - (d_weights * weights).sum(-1, keepdim=True))
d_scores = weights.detach() * (d_weights - (d_weights * weights.detach()).sum(dim=-1, keepdim=True))
print(f"d_loss/d_scores (through softmax Jacobian): shape {d_scores.shape}")

# d_loss/d_Q = d_scores @ K / sqrt(d)
d_Q = d_scores @ K_ag.detach() / (d_model ** 0.5)
print(f"d_loss/d_Q: shape {d_Q.shape}")

# d_loss/d_K = d_scores^T @ Q / sqrt(d)
d_K = d_scores.transpose(-2, -1) @ Q_ag.detach() / (d_model ** 0.5)
print(f"d_loss/d_K: shape {d_K.shape}")

print("\nGradients through projections:")
# d_loss/d_W_q = x^T @ d_Q
d_W_q_manual = x.squeeze(0).T @ d_Q.squeeze(0)
print(f"d_loss/d_W_q: shape {d_W_q_manual.shape}, norm {d_W_q_manual.norm():.6f}")

In [ ]:
# Autograd backward
loss.backward()

print("=== Verification: Manual vs Autograd ===\n")

# Compare gradients
for name, manual, autograd in [
    ("d_loss/d_W_q", d_W_q_manual, W_q_ag.grad),
    ("d_loss/d_V (from attention)", d_V.squeeze(0), None),  # V grad is more complex
]:
    if autograd is not None:
        diff = (manual - autograd).abs().max().item()
        match = "MATCH" if diff < 1e-4 else f"DIFF={diff:.6f}"
        print(f"  {name}: {match}")
        print(f"    Manual norm: {manual.norm():.6f}")
        print(f"    Autograd norm: {autograd.norm():.6f}")

# Full verification with torch.autograd.gradcheck would be:
print("\nKey insight: autograd computes the exact same chain rule we did manually.")
print("Understanding this is important for:")
print("  - Debugging training (vanishing/exploding gradients)")
print("  - Custom loss functions and architectures")
print("  - Understanding why certain architectures train better (gradient flow)")

## Softmax Jacobian Derivation

The softmax function: $s_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$

The Jacobian $\frac{\partial s_i}{\partial z_j}$:

**Case i = j:**
$$\frac{\partial s_i}{\partial z_i} = s_i(1 - s_i)$$

**Case i ≠ j:**
$$\frac{\partial s_i}{\partial z_j} = -s_i \cdot s_j$$

**In matrix form:**
$$\frac{\partial \mathbf{s}}{\partial \mathbf{z}} = \text{diag}(\mathbf{s}) - \mathbf{s}\mathbf{s}^T$$

**Why this matters for attention:**
- The softmax Jacobian determines how gradients flow through attention weights
- When attention is very peaked (one large weight), gradients primarily flow through that connection
- When attention is uniform, gradients are distributed evenly
- This is why temperature scaling in attention ($QK^T / \sqrt{d}$) is crucial: it controls gradient flow

## System Design: Video Inference Serving

```
┌─────────────────────────────────────────────────────┐
│                 API Gateway (K8s Ingress)            │
│                 Rate limiting, auth, routing          │
└─────────────┬───────────────────────────┬───────────┘
              │                           │
    ┌─────────▼─────────┐      ┌─────────▼─────────┐
    │  Prompt Service    │      │  Status Service    │
    │  - Text encode     │      │  - Job tracking    │
    │  - Safety filter   │      │  - Progress SSE    │
    │  CPU only, fast    │      │  Redis-backed      │
    └─────────┬─────────┘      └───────────────────┘
              │
    ┌─────────▼─────────┐
    │   Job Queue        │
    │   (Redis/SQS)      │
    │   Priority levels  │
    └─────────┬─────────┘
              │
    ┌─────────▼──────────────────────────────────────┐
    │         GPU Worker Pool (K8s, autoscaled)       │
    │  ┌──────────┐ ┌──────────┐ ┌──────────┐       │
    │  │ Worker 1 │ │ Worker 2 │ │ Worker N │       │
    │  │ A100/H100│ │ A100/H100│ │ A100/H100│       │
    │  │          │ │          │ │          │       │
    │  │ Model    │ │ Model    │ │ Model    │       │
    │  │ (sharded)│ │ (sharded)│ │ (sharded)│       │
    │  └────┬─────┘ └────┬─────┘ └────┬─────┘       │
    │       └─────────────┴─────────────┘             │
    │              Shared model cache                  │
    │              (safetensors on NFS/S3)              │
    └─────────┬──────────────────────────────────────┘
              │
    ┌─────────▼─────────┐
    │  Output Pipeline   │
    │  - Video encode    │
    │  - Watermark       │
    │  - Upload to CDN   │
    └───────────────────┘
```

**Key decisions:**
- **Async by default**: video gen takes seconds-minutes, can't block HTTP requests
- **Model sharding**: large models (10B+ params) need tensor parallelism across GPUs
- **Autoscaling**: K8s HPA on GPU utilization + queue depth
- **Model caching**: keep weights in shared storage, workers load on startup
- **Streaming for AR**: if using AR generation, stream frames to client via WebSocket/SSE

**Production patterns:**
- Kubernetes-based orchestration (not SLURM) for serving workloads
- Custom PyTorch Lightning variants common for training
- Lance for data (fast random access for training, could also cache intermediate results)

## System Design: End-to-End Video Data Pipeline

**Prompt:** "Design a data pipeline that takes raw video from the internet and produces training-ready data for a video diffusion model."

```
┌─────────────────────────────────────────────────────┐
│ Orchestration: Flyte (K8s-native, typed, cached)     │
│ Compute: Ray Data on Anyscale/GKE                    │
│ Storage: Lance lakehouse (canonical) + WebDataset     │
│          shards (training snapshots)                  │
├─────────────────────────────────────────────────────┤
│ Stage 1: Ingestion                                   │
│   Sources: web crawl, licensed datasets, partnerships│
│   Tools: yt-dlp, custom scrapers, S3 bulk transfer   │
│   Output: raw video files in object storage (S3)     │
│   Monitoring: ingestion rate, source diversity        │
├─────────────────────────────────────────────────────┤
│ Stage 2: Deduplication                               │
│   Method: perceptual hashing (dHash) for near-exact, │
│           embedding cosine sim for semantic dedup     │
│   Scale: ~O(N log N) with LSH indexing               │
│   Compute: Ray Data on CPU pool                      │
│   Failure mode: over-dedup removes valid variations  │
│   Monitoring: dedup rate, false positive sampling     │
├─────────────────────────────────────────────────────┤
│ Stage 3: Scene Segmentation                          │
│   Tool: PySceneDetect (content-aware detection)      │
│   Output: individual scenes (2-30s clips)            │
│   Compute: Ray Data on CPU pool                      │
│   Failure mode: soft transitions missed → tune       │
│     threshold per content type                       │
│   Monitoring: avg scene length distribution          │
├─────────────────────────────────────────────────────┤
│ Stage 4: Hierarchical Quality Filtering (5 stages)   │
│   4a. Resolution filter: drop < 720p       (CPU)    │
│   4b. Aspect ratio filter: drop extreme    (CPU)    │
│   4c. Technical quality: sharpness, noise  (CPU)    │
│   4d. Aesthetic score: CLIP-based          (GPU)    │
│   4e. Safety filter: NSFW, watermark det   (GPU)    │
│                                                       │
│   Cheapest filters first → reduce GPU load by 5-10× │
│   Compute: Ray Data with GPU actor pools for 4d/4e  │
│   Typical survival rate: 10-15% of raw data.         │
│   Monitoring: per-stage pass rate, quality score dist │
├─────────────────────────────────────────────────────┤
│ Stage 5: Foundation Model Auto-Labeling              │
│   5a. VLM re-captioning (Qwen-VL 7B, batched)      │
│       → structured captions: scene, motion, camera   │
│   5b. SAM segmentation → object masks               │
│   5c. CLIP embedding → semantic vectors              │
│   5d. Consensus scoring across models                │
│   Compute: Ray Serve with dedicated GPU actor pools  │
│   Separate from training GPU allocation              │
│   Failure mode: VLM hallucination → CLIP score gate  │
│   Monitoring: caption diversity, CLIP alignment dist │
├─────────────────────────────────────────────────────┤
│ Stage 6: Data Mixing & Curriculum                    │
│   Mix ratios: high-quality (40%), medium (30%),      │
│     synthetic (20%), instruction pairs (10%)         │
│   Curriculum: images first → short video → long      │
│   Progressive thresholds per training stage:         │
│     early = loose filters, late = strict filters     │
│   Monitoring: category distribution, training loss   │
├─────────────────────────────────────────────────────┤
│ Stage 7: Store to Feature Lakehouse                  │
│   Format: Lance tables (canonical, queryable)        │
│   Schema: video_path, caption, embeddings,           │
│     quality_scores, motion_scores, metadata          │
│   Schema evolution: append new embedding columns     │
│     without rewriting existing data                  │
│   Training snapshot: materialize WebDataset shards   │
│     for frozen training runs (1GB tar each)          │
│   Monitoring: shard size uniformity, read throughput  │
└─────────────────────────────────────────────────────┘
```

**Key design decisions:**
- **Lakehouse + shards coexist.** Lance is the living canonical store (queryable, evolvable). WebDataset shards are frozen snapshots for large distributed training runs. Both are needed.
- **Ray Data + Ray Serve for compute.** CPU tasks (dedup, scene detect, resolution filter) run on Ray Data CPU pools. GPU tasks (VLM captioning, aesthetic scoring, embedding) run on Ray Serve actor pools with long-lived model processes. Training GPUs stay 100% on gradient computation.
- **Flyte for orchestration.** K8s-native scheduling, typed inputs/outputs, built-in caching. 10K+ parallel map tasks per stage. Retries and idempotency are first-class.
- **Cheapest filter first.** Resolution check (CPU, microseconds) before aesthetic scoring (GPU, milliseconds). 5-10× total compute savings.
- **Schema evolution.** Swap CLIP for SigLIP? Append a new embedding column. Don't rewrite petabytes.
- **Curriculum = progressive thresholds.** The same lakehouse serves different training stages. Early training queries with loose filters; late training queries with strict filters. This IS the curriculum.

**Scale estimate for 100M raw videos → ~10M training clips:**
- Ingestion: ~1PB raw storage
- Dedup + segmentation: ~5000 CPU-hours (Ray Data)
- Quality filtering: ~2000 CPU-hours + ~500 GPU-hours (Ray Serve actor pools)
- Auto-labeling (VLM + CLIP + SAM): ~2500 GPU-hours
- Lance lakehouse + WebDataset sharding: ~500 CPU-hours
- Total: ~3000 GPU-hours + ~7500 CPU-hours
- Timeline: 1-2 weeks with a 50-GPU cluster on Anyscale

## System Design: Re-caption 2B Video Clips at Scale

**Prompt:** "You have 2 billion video clips with noisy alt-text. Design the system to re-caption them all with a VLM."

```
Scale Analysis
├── 2B clips × ~5 frames sampled per clip = 10B VLM inference calls
├── VLM throughput (7B model, A100): ~50 clips/sec with batching
├── Single GPU time: 2B / 50 = 40M seconds ≈ 463 GPU-days
├── Target: complete in 2 weeks → need ~33 A100s sustained
└── Cost estimate: 33 GPUs × 14 days × $2/GPU-hr ≈ $22K (cloud)

Architecture:
┌─────────────────────────────────────────────────────┐
│                  Orchestrator (Airflow/Prefect)      │
│  - Job scheduling, retry logic, progress tracking    │
│  - Split 2B clips into ~2000 shards of 1M each     │
└─────────────┬───────────────────────────────────────┘
              │
┌─────────────▼───────────────────────────────────────┐
│           Frame Sampler (CPU fleet, K8s jobs)        │
│  - Sample N frames per clip (uniform or keyframe)    │
│  - Resize to VLM input resolution (e.g., 384×384)   │
│  - Write frame batches to staging storage            │
│  Throughput: ~2000 clips/sec per CPU pod             │
└─────────────┬───────────────────────────────────────┘
              │
┌─────────────▼───────────────────────────────────────┐
│          VLM Inference Fleet (GPU, autoscaled)       │
│  ┌──────────────────────────────────────────────┐   │
│  │ vLLM / TGI serving with continuous batching   │   │
│  │ Model: Qwen-VL-7B or InternVL-7B (open)      │   │
│  │ Batch size: 32-64 (depending on VRAM)         │   │
│  │ Prompt: structured caption template           │   │
│  └──────────────────────────────────────────────┘   │
│  Replicas: 30-50 pods, each with 1× A100            │
│  Autoscale on queue depth                            │
└─────────────┬───────────────────────────────────────┘
              │
┌─────────────▼───────────────────────────────────────┐
│          Quality Filter (CPU fleet)                  │
│  - Hallucination detection:                          │
│    • CLIP score between caption and frames           │
│    • Flag captions with low alignment (<0.25)        │
│  - Format validation: structured fields present?     │
│  - Dedup: exact duplicate captions across clips      │
│  - Quality sampling: human review 0.01% random       │
└─────────────┬───────────────────────────────────────┘
              │
┌─────────────▼───────────────────────────────────────┐
│          Caption Store (object storage + index)      │
│  - Parquet files: clip_id, caption, quality_score    │
│  - Indexed by clip_id for fast join with video       │
│  - Versioned: keep old captions for A/B comparison   │
└─────────────────────────────────────────────────────┘
```

**Cost optimization levers:**
- **Model size:** 7B VLM is 4× faster than 72B, quality difference is often small for structured captions
- **Quantization:** INT8 inference gives ~1.5× throughput with minimal quality loss
- **Spot instances:** 60-70% cost reduction, but need checkpoint/retry logic
- **Batch size:** Maximize GPU utilization via continuous batching (vLLM)
- **Frame sampling:** 3 frames vs 8 frames per clip — benchmark quality vs throughput

**Hallucination detection is critical:**
- VLMs hallucinate objects, actions, and scene details
- CLIP score threshold filters the worst cases
- Cross-check: if 2 different VLMs agree, caption is likely accurate
- Human sampling rate: 0.01% = 200K manual checks on 2B clips (budget ~$20K at $0.10/check)

**Failure modes:**
- GPU OOM on long/complex frames → set max output tokens, truncate gracefully
- VLM generates repetitive/template captions → diversity filter, temperature tuning
- Network bottleneck reading frames from storage → co-locate compute and storage
- Stale queue entries after pod restart → idempotent processing, dedup on write

In [ ]:
# EXERCISE: System Design as Code — Re-captioning Pipeline
#
# You're designing a pipeline to re-caption 2B video clips using a VLM.
# Write the orchestration pseudocode as real Python classes/functions.
# Focus on: batching strategy, fault tolerance, throughput estimation.

@dataclass
class RecaptionConfig:
    """Configuration for a large-scale re-captioning job."""
    total_clips: int = 2_000_000_000
    vlm_throughput_per_gpu: float = 50.0  # clips/sec per GPU
    num_gpus: int = 64
    batch_size: int = 32
    max_retries: int = 3
    checkpoint_every_n: int = 100_000


def estimate_job_duration(config: RecaptionConfig) -> dict:
    """Estimate wall-clock time and cost for the re-captioning job.

    Returns:
        dict with: total_batches, throughput_clips_per_sec,
        estimated_hours, gpu_hours
    """
    # YOUR CODE
    raise NotImplementedError


def process_batch(clip_ids: list[int], vlm_fn, config: RecaptionConfig) -> list[dict]:
    """Process a batch of clips: load, caption, validate, return results.

    Should handle failures gracefully (retry individual clips, log failures).

    Args:
        clip_ids: List of clip IDs to process.
        vlm_fn: Callable that takes a clip and returns a caption string.
        config: Job configuration.

    Returns:
        List of {"clip_id": int, "caption": str, "status": str} dicts.
    """
    # YOUR CODE
    raise NotImplementedError


# Test your estimates
config = RecaptionConfig()
estimates = estimate_job_duration(config)
print(f"Total clips: {config.total_clips:,}")
print(f"GPUs: {config.num_gpus}")
print(f"Throughput: {estimates['throughput_clips_per_sec']:,.0f} clips/sec")
print(f"Estimated time: {estimates['estimated_hours']:.1f} hours")
print(f"GPU-hours: {estimates['gpu_hours']:,.0f}")

## System Design: Action-Conditioned World Model Data Pipeline

**Prompt:** "Design the data pipeline for training an action-conditioned world model — given an action (joystick input, robot command), predict the next video frame(s)."

```
Data Sources
├── Simulator environments (Unity, Unreal, Isaac Sim)
│   → Deterministic replay, unlimited volume, cheap
│   → Sim-to-real gap is the fundamental problem
├── Robot teleoperation logs
│   → Real-world, expensive, limited volume
│   → Action format: joint angles, end-effector poses, force/torque
├── Game replay data (licensed)
│   → Rich action space, diverse environments
│   → Action format: discrete buttons + analog sticks
└── Dashcam / ego-vehicle data
    → Steering + acceleration as continuous actions
    → Massive volume but limited action diversity

Pipeline:
┌──────────────────────────────────────────┐
│ 1. Action-Video Alignment               │
│    - Synchronize action timestamps with  │
│      video frames (±16ms tolerance)      │
│    - Interpolate actions to video FPS    │
│    - Validate: action at t → visual     │
│      change at t+1 must be causal       │
├──────────────────────────────────────────┤
│ 2. Temporal Segmentation                 │
│    - Split into action-conditioned       │
│      episodes (start → goal state)       │
│    - Filter: discard static segments     │
│      where action has no visual effect   │
│    - Annotate: success/failure labels    │
├──────────────────────────────────────────┤
│ 3. Action Normalization                  │
│    - Map diverse action formats to       │
│      unified representation              │
│    - Continuous: normalize to [-1, 1]    │
│    - Discrete: one-hot or learned embed  │
│    - Multi-modal: separate streams       │
├──────────────────────────────────────────┤
│ 4. Sim-to-Real Augmentation              │
│    - Domain randomization on sim data    │
│      (lighting, textures, physics params)│
│    - Style transfer: sim → photorealistic│
│    - Mix ratio: typically 80% sim + 20%  │
│      real, anneal toward real over time  │
├──────────────────────────────────────────┤
│ 5. Causal Validation                     │
│    - Counterfactual check: same state +  │
│      different action → different outcome│
│    - Filter sequences where actions have │
│      no visible effect (camera static)   │
│    - Physics plausibility scoring        │
├──────────────────────────────────────────┤
│ 6. Sharding for AR Training              │
│    - Sequential access pattern (not      │
│      random) — shard by episode          │
│    - Include action tensor alongside     │
│      video frames in each sample         │
│    - WebDataset: action.npy + video.mp4  │
│      per sample in tar                   │
└──────────────────────────────────────────┘
```

**Key tradeoffs:**
- **Sim volume vs. real quality:** Sim data is unlimited but has distribution gap. Real data is expensive but trains better models. Start with sim, finetune on real.
- **Action granularity:** Too coarse (discrete) limits model expressiveness. Too fine (raw motor commands) makes learning harder. Find the right abstraction level for the task.
- **Episode length:** Longer episodes capture more complex behaviors but are harder to learn from. Start with short episodes (2-4s), curriculum to longer.

**Scale estimate for 10M episodes:**
- Sim generation: ~500 GPU-hours (parallelizable across sim instances)
- Action alignment & validation: ~200 CPU-hours
- Augmentation: ~1000 GPU-hours
- Total storage: ~50TB (video + action tensors)

## Self-Assessment Cards
*(Read question, try to answer, then check the answer)*

---

**Q: Walk through the video diffusion pipeline end-to-end.**

<details>
<summary>Answer</summary>

Text → CLIP/T5 text encoder → text embeddings. Random noise tensor [B, C, T, H, W] in latent space. For each denoising step: concatenate conditional and unconditional batch (for CFG), run through 3D UNet or DiT (with temporal attention layers), get noise prediction, apply CFG weighting, scheduler step to get less-noisy latent. After all steps: decode latent → pixel space with VAE (frame by frame or 3D VAE). Post-process: interpolate frames if needed, encode to MP4.
</details>

---

**Q: How do you take a model checkpoint to production?**

<details>
<summary>Answer</summary>

(1) Validate: eval metrics on held-out test set, compare to baseline. (2) Optimize: FP16/BF16 quantization, torch.compile, choose fast scheduler (DPM-Solver++ 20 steps). (3) Serve: wrap in async serving framework (Triton, vLLM for AR), configure batching, set up health checks. (4) Deploy: canary rollout (5% traffic), monitor latency p50/p95/p99, VRAM usage, error rate. (5) Guardrails: input validation, output safety filtering, rate limiting. (6) Observe: log prompt/output pairs for quality monitoring, set up alerts on metric degradation.
</details>

---

**Q: New model is 3x slower than target. What do you do?**

<details>
<summary>Answer</summary>

(1) Profile first: is it compute-bound (need fewer FLOPs) or memory-bound (need less data movement)? Use torch.profiler or nsight. (2) Low-hanging fruit: FP16 (~2x), torch.compile (~1.3x), better scheduler (50→20 steps = 2.5x). (3) Attention: Flash Attention, STA for video. (4) Distillation: if the above isn't enough, distill to fewer steps (LCM, consistency models). (5) Architecture: quantization (INT8/INT4), pruning, smaller model with distillation. Always benchmark quality alongside -- optimization budget depends on acceptable quality loss.
</details>

---

**Q: Diffusion vs autoregressive -- when to use which?**

<details>
<summary>Answer</summary>

**Diffusion (bidirectional):** All frames generated simultaneously. High quality because every frame sees every other frame during denoising. Best for: fixed-length, high-quality output. Limitation: can't stream, O(N²) attention over all frames.

**Autoregressive (causal):** Frames generated sequentially, each conditioned on previous. Best for: streaming/interactive, variable length, world models (next-state prediction). Limitation: error accumulation, sequential bottleneck.

**Hybrid A2D approach:** AR for temporal/semantic (what happens next), diffusion for visual fidelity (how it looks). Gets streaming from AR + quality from diffusion.
</details>

---

**Q: How do you evaluate video model quality?**

<details>
<summary>Answer</summary>

**Automated:** FID/FVD (distribution quality), CLIP score (text alignment), LPIPS (perceptual quality), temporal consistency metrics (optical flow smoothness between frames). **Human eval:** Side-by-side comparisons on curated prompt set, rating on: visual quality, temporal coherence, prompt adherence, artifact presence. **Key insight:** No single metric captures video quality. FVD is the standard but high-variance and reference-dependent. In practice: automated metrics gate bad models, human eval picks the best.
</details>

---

**Q: Design a data pipeline for 30M videos.**

<details>
<summary>Answer</summary>

**Ingest:** Download/crawl → deduplicate (perceptual hashing) → store raw in object storage (S3). **Process:** Scene detection (PySceneDetect) → quality filter (resolution, motion, aesthetics classifier) → caption (VLM re-captioning at scale, GPU batch job) → embed (CLIP/video encoder). **Store:** Pack into Lance format (fast random access, native video ops, vector index). **Serve:** Lance-backed DataLoader with global shuffling, dynamic batching by aspect ratio/duration. **Scale:** Distribute processing with K8s jobs, ~1000 GPU-hours for captioning 30M videos. Monitor: data quality dashboards, embedding distribution drift detection.
</details>

---

**Q: You need to re-caption 2B video clips. Walk through the architecture and cost analysis.**

<details>
<summary>Answer</summary>

**Scale math:** 2B clips at ~50 clips/sec per A100 (7B VLM, batched) = 463 GPU-days. Target 2 weeks → 33 A100s sustained, ~$22K cloud cost. **Architecture:** Orchestrator (Airflow) splits into 2000 shards → CPU fleet samples frames → GPU fleet runs VLM inference (vLLM with continuous batching) → quality filter (CLIP score for hallucination detection, format validation) → caption store (Parquet, versioned). **Key optimizations:** Use 7B not 72B VLM (4× faster, similar quality for structured captions), INT8 quantization (1.5× throughput), spot instances (60-70% cost savings with retry logic). **Critical:** Hallucination detection — CLIP score threshold + 0.01% human review sampling.
</details>

---

**Q: How would you design the data pipeline for an action-conditioned world model?**

<details>
<summary>Answer</summary>

**Data sources:** Simulator environments (cheap, unlimited, but sim-to-real gap), robot teleoperation (expensive, real), game replays (rich actions), dashcam (massive volume). **Pipeline:** (1) Action-video temporal alignment (±16ms tolerance, interpolate to video FPS). (2) Episode segmentation (action start → goal state). (3) Action normalization (unified format across sources). (4) Sim-to-real augmentation (domain randomization, style transfer, anneal mix ratio). (5) Causal validation (counterfactual: different action → different outcome). (6) Shard by episode for sequential AR training access. **Key tradeoff:** Start 80% sim / 20% real, anneal toward real. Episode length curriculum: short (2-4s) first.
</details>

---

**Q: Why does ordering quality filters from cheapest to most expensive matter? Give concrete numbers.**

<details>
<summary>Answer</summary>

**Filter cost hierarchy:** Resolution check (~0.001ms, CPU) → aspect ratio (~0.001ms) → sharpness/noise (~1ms, CPU) → aesthetic score (~10ms, GPU) → safety/NSFW (~10ms, GPU). If 100M videos enter the pipeline and resolution filter removes 40%, that's 40M fewer videos hitting the expensive GPU filters. **Concrete savings:** Running aesthetic scoring on 100M vs 60M videos at 10ms/video = 1000 GPU-hours saved. Running NSFW detection on 100M vs 50M (after resolution + aspect ratio filters) = 500 GPU-hours saved. **Total:** ordering saves ~1500 GPU-hours, which at $2/GPU-hr = $3000 per pipeline run. Over multiple iterations of threshold tuning, this compounds.
</details>

## MoE for Diffusion Models

Mixture of Experts in diffusion (Wan2.2 architecture):
- Standard transformer blocks with MoE FFN layers
- Each token routed to top-k experts (typically 2 of 8-16 total)
- Benefit: much larger total parameter count with same inference cost (only k experts active)
- For video: different experts can specialize in different types of content (motion, static, transitions)
- Wan2.2 uses MoE to scale to larger effective model size without proportional compute increase

## Open-Source Video Model Landscape (2025-2026)

| Model | Org | Params | Architecture | Key Innovation |
|-------|-----|--------|-------------|----------------|
| Wan2.1 | Alibaba | 1.3B-14B | DiT + 3D VAE | Open weights, scalable |
| Wan2.2 | Alibaba | 14B MoE | DiT + MoE FFN | MoE for efficiency |
| HunyuanVideo | Tencent | 13B | Dual-stream DiT | Text+video dual stream |
| Mochi 1 | Genmo | ~10B | Asymmetric DiT | Asymmetric attention |
| LTX-Video 2 | Lightricks | 14B+5B | Dual-stream DiT | Native audio (5B stream) |
| CogVideoX | Zhipu | 2B-5B | 3D attention DiT | Expert transformer |
| Open-Sora | HPC-AI | Varies | STDiT | Open replication of Sora |

**Proprietary models** (Gen-3, Gen-4, Gen-4.5) are differentiated by:
- AR + diffusion hybrid (A2D architecture)
- Native audio generation
- Real-time interaction capability (world models)
- GWM-1: fully AR, action-conditioned

## Industry Context

**NVIDIA Partnership:**
- NVIDIA partnered on Cosmos (world foundation models) with video generation companies
- Blackwell GPU architecture optimized for video generation workloads
- Vera Rubin (next-gen) will push real-time video generation further

**The competitive landscape:**
- OpenAI (Sora): massive compute, less open
- Google (Veo/Lumiere): research-heavy, integrated with Gemini
- AR-focused labs: production-focused, real-time emphasis, strong AR research teams
- Alibaba (Wan): open-source leader, fast iteration
- Kling, Minimax, Hailuo: Chinese competitors with rapid progress

**Key differentiators across companies:**
- Real-time / interactive generation (unique to AR approaches)
- Production deployment experience (iterating through multiple model generations)
- Creative tool integration (not just API, full creative suite)
- Audio-visual joint generation